SMS Spam Detection Using TF-IDF and Machine Learning

Objective

To develop a text classification model that can effectively classify spam emails using TF-IDF vectorization and supervised machine learning algorithms, it is necessary to compare the performance of the models based on precision, recall, and F1-score evaluation metrics

In [1]:
import pandas as pd
df=pd.read_csv("mail_data.csv")
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
df.shape

(5572, 2)

In [3]:
df.describe()

,Category,Message
count,5572,5572
unique,2,5157
top,ham,"Sorry, I'll call later"
freq,4825,30


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5572 non-null   object
 1   Message   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [5]:
df.duplicated().sum()

415

In [6]:
df = df.drop_duplicates()

In [7]:
df.duplicated().sum()

0

In [8]:
df.shape

(5157, 2)

In [9]:
df["Category"].value_counts()

Category
ham     4516
spam     641
Name: count, dtype: int64

Class imbalance was handled by maintaining class distributions through stratified splitting and using class weights in Logistic Regression. Rather than using oversampling techniques such as SMOTE, which may affect sparse TF-IDF representation, model evaluation was based on precision, recall, and F1-score instead of accuracy

In [10]:
df.isnull().sum()

Category    0
Message     0
dtype: int64

In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Category"] = le.fit_transform(df["Category"])  

In [12]:
df.head()

,Category,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000,stop_words="english")

TfidfVectorizer is used to transform text data to numerical representations using TF-IDF features. Top 5000 words were used based on their frequency to control the dimensions and overfitting. Also, English stop words were removed to filter out uninformative words.

In [14]:
X = tfidf.fit_transform(df["Message"])
y = df["Category"]

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,
    random_state=42,stratify=y)

## LogisticRegression

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

lr = LogisticRegression(class_weight="balanced", max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [17]:
print("Logistic Regression Results\n")
print(classification_report(y_test, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

Logistic Regression Results

              precision    recall  f1-score   support

           0       0.98      0.99      0.98       904
           1       0.91      0.86      0.88       128

    accuracy                           0.97      1032
   macro avg       0.94      0.92      0.93      1032
weighted avg       0.97      0.97      0.97      1032

Confusion Matrix:
 [[893  11]
 [ 18 110]]


## MultinomialNB

In [18]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

In [19]:
print("Naive Bayes Results\n")
print(classification_report(y_test, y_pred_nb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))

Naive Bayes Results

              precision    recall  f1-score   support

           0       0.97      1.00      0.98       904
           1       1.00      0.77      0.87       128

    accuracy                           0.97      1032
   macro avg       0.98      0.88      0.93      1032
weighted avg       0.97      0.97      0.97      1032

Confusion Matrix:
 [[904   0]
 [ 30  98]]


## RandomForestClassifier

In [20]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [21]:
print("Random Forest Results\n")
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

Random Forest Results

              precision    recall  f1-score   support

           0       0.97      1.00      0.98       904
           1       0.98      0.77      0.86       128

    accuracy                           0.97      1032
   macro avg       0.97      0.88      0.92      1032
weighted avg       0.97      0.97      0.97      1032

Confusion Matrix:
 [[902   2]
 [ 30  98]]


Logistic Regression, Multinomial Naive Bayes, and Random Forest algorithms were tested on spam classification using TF-IDF features. Although all models had similar overall accuracy (97%), accuracy was not the main focus because of the class imbalance problem. Logistic Regression offered the best trade-off between precision (0.91) and recall (0.86) for the spam class, resulting in the highest F1-score among all models. Naive Bayes and Random Forest had high precision but poor recall, missing more spam emails. As the problem of spam classification is more critical and has higher priority than just focusing on accuracy, Logistic Regression was chosen as the final model for this project